## 21. اعتبارسنجی مختصات و نقشه

پیش از ساخت نقشه:

- Missing coordinates را گزارش کنید.
- Latitude و Longitude جابه‌جا نشده باشند.
- نقاط خارج از محدوده ایران شناسایی شوند.
- نقاط خارج از محدوده شهر بررسی شوند.
- تعداد مختصات تکراری غیرعادی بررسی شود.
- دقت مکانی و احتمال ناشناس‌سازی مختصات توضیح داده شود.
- از نمایش نقطه‌ای یک میلیون رکورد خودداری شود.

روش‌های مناسب‌تر:

- Aggregation محله‌ای
- Hexbin
- Grid aggregation
- Sample کنترل‌شده
- Choropleth در صورت وجود مرزهای معتبر

In [1]:
import pandas as pd
import geopandas as gpd
import folium
from folium import Circle, CircleMarker
from shapely.geometry import Point

In [2]:
df = pd.read_feather("../Outputs/19_df.feather")



<div dir="rtl" align="right">

این بخش با AI زده شده است و خروجی ان توسط تیم بررسی شده است
</div>

In [ ]:
# =========================================================
# 1. تبدیل مختصات به عدد
# =========================================================

df['location_latitude'] = pd.to_numeric(
    df['location_latitude'],
    errors='coerce'
)

df['location_longitude'] = pd.to_numeric(
    df['location_longitude'],
    errors='coerce'
)

df['location_radius'] = pd.to_numeric(
    df['location_radius'],
    errors='coerce'
)


# =========================================================
# 2. فقط مختصات کامل
# =========================================================

valid_coords = (
    df['location_latitude'].notna() &
    df['location_longitude'].notna()
)

geo_df = df.loc[valid_coords].copy()


# =========================================================
# 3. ساخت GeoDataFrame
# =========================================================

geo_df = gpd.GeoDataFrame(
    geo_df,
    geometry=gpd.points_from_xy(
        geo_df['location_longitude'],
        geo_df['location_latitude']
    ),
    crs='EPSG:4326'
)


# =========================================================
# 4. دریافت مرز ایران
# =========================================================
url = "https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_IRN_0.json"
iran_gdf = gpd.read_file(url)

# =========================================================
# 5. ساخت Buffer دو کیلومتری اطراف ایران
# =========================================================

# تبدیل به سیستم مختصات متری
iran_projected = iran_gdf.to_crs('EPSG:32639')

# 5 کیلومتر Buffer
iran_buffer = iran_projected.geometry.buffer(5000)

# تبدیل دوباره به WGS84
iran_buffer_wgs84 = gpd.GeoDataFrame(
    geometry=iran_buffer,
    crs='EPSG:32639'
).to_crs('EPSG:4326')


# =========================================================
# 6. بررسی داخل محدوده ایران + Buffer
# =========================================================

iran_area = iran_buffer_wgs84.geometry.iloc[0]

geo_df['inside_iran'] = geo_df.geometry.within(
    iran_area
)

geo_df['outside_iran'] = ~geo_df['inside_iran']


# =========================================================
# 7. نقاط خارج از ایران
# =========================================================

outside = geo_df[
    geo_df['outside_iran']
].copy()


# ایجاد ستون Flag
df['coordinates_outlier'] = False

# Flag کردن نقاط خارج از محدوده ایران
df.loc[
    outside.index,
    'coordinates_outlier'
] = True


print("========================================")
print("Coordinate Validation")
print("========================================")

print(
    "تعداد کل رکوردها:",
    len(df)
)

print(
    "مختصات Missing:",
    len(df) - len(geo_df)
)

print(
    "مختصات معتبر:",
    len(geo_df)
)

print(
    "خارج از ایران:",
    len(outside)
)
print(df['coordinates_outlier'].sum())



Coordinate Validation
تعداد کل رکوردها: 999953
مختصات Missing: 344367
مختصات معتبر: 655586
خارج از ایران: 1022
1022

نقشه ذخیره شد: iran_coordinate_validation.html


کد نمایش دادن داده های پرت بر روی نقشه

In [ ]:

# # =========================================================
# # 8. ساخت نقشه
# # =========================================================

# m = folium.Map(
#     location=[32.0, 53.0],
#     zoom_start=5,
#     tiles='OpenStreetMap'
# )


# # =========================================================
# # 9. نمایش مرز واقعی ایران
# # =========================================================

# iran_border_wgs84 = iran_gdf.to_crs('EPSG:4326')

# folium.GeoJson(
#     iran_border_wgs84.to_json(),
#     name='Iran Actual Border',
#     style_function=lambda x: {
#         'fillColor': 'none',
#         'color': 'blue',
#         'weight': 2
#     }
# ).add_to(m)


# # =========================================================
# # 10. نمایش Buffer ساحلی
# # =========================================================

# folium.GeoJson(
#     iran_buffer_wgs84.to_json(),
#     name='Iran + 2km Coastal Buffer',
#     style_function=lambda x: {
#         'fillColor': 'none',
#         'color': 'green',
#         'weight': 1,
#         'dashArray': '5, 5'
#     }
# ).add_to(m)


# # =========================================================
# # 11. نمایش نقاط خارج از محدوده
# # =========================================================

# for idx, row in outside.iterrows():

#     lat = row['location_latitude']
#     lon = row['location_longitude']

#     popup_text = f"""
#     <b>Index:</b> {idx}<br>
#     <b>City:</b> {row.get('city_slug', '')}<br>
#     <b>Neighborhood:</b> {row.get('neighborhood_slug', '')}<br>
#     <b>Category:</b> {row.get('cat2_slug', '')}<br>
#     <b>Latitude:</b> {lat}<br>
#     <b>Longitude:</b> {lon}<br>
#     <b>Radius:</b> {row.get('location_radius', '')} m
#     """

#     # نقطه
#     CircleMarker(
#         location=[
#             float(lat),
#             float(lon)
#         ],
#         radius=5,
#         popup=folium.Popup(
#             popup_text,
#             max_width=400
#         ),
#         tooltip=f"Index: {idx}",
#         color='red',
#         fill=True
#     ).add_to(m)

#     # =====================================================
#     # نمایش location_radius
#     # =====================================================

#     radius = row['location_radius']

#     if pd.notna(radius) and radius > 0:

#         Circle(
#             location=[
#                 float(lat),
#                 float(lon)
#             ],
#             radius=float(radius),
#             color='red',
#             fill=False,
#             weight=1
#         ).add_to(m)


# # =========================================================
# # 12. Layer Control
# # =========================================================

# folium.LayerControl().add_to(m)


# # =========================================================
# # 13. ذخیره نقاط پرت بر روی نقشه
# # =========================================================

# output_file = '../Outputs/iran_coordinate_validation.html'

# m.save(output_file)

# print(
#     f"\nنقشه ذخیره شد: {output_file}"
# )

In [4]:
df.columns

Index(['cat2_slug', 'cat3_slug', 'city_slug', 'neighborhood_slug',
       'created_at_month', 'user_type', 'description', 'title', 'rent_mode',
       'rent_value', 'rent_to_single', 'rent_type', 'price_mode',
       'price_value', 'credit_mode', 'credit_value', 'rent_credit_transform',
       'transformable_price', 'transformable_credit', 'transformed_credit',
       'transformable_rent', 'transformed_rent', 'land_size', 'building_size',
       'deed_type', 'has_business_deed', 'floor', 'rooms_count',
       'total_floors_count', 'unit_per_floor', 'has_balcony', 'has_elevator',
       'has_warehouse', 'has_parking', 'construction_year', 'is_rebuilt',
       'has_water', 'has_warm_water_provider', 'has_electricity', 'has_gas',
       'has_heating_system', 'has_cooling_system', 'has_restroom',
       'has_security_guard', 'has_barbecue', 'building_direction', 'has_pool',
       'has_jacuzzi', 'has_sauna', 'floor_material', 'property_type',
       'regular_person_capacity', 'extra_person

In [5]:
df.to_feather("../Outputs/21_df.feather")